# Interactive Case Study: Policy Gradient Algorithms in PyTorch

This notebook provides a complete hands-on implementation and performance comparison of the foundational policy gradient algorithms explained in Lecture 9 and Lecture 10:
1. **REINFORCE (Monte Carlo Policy Gradient)**
2. **REINFORCE with Baseline**
3. **One-Step Actor-Critic (TD)**
4. **Advantage Actor-Critic (A2C)** (Synchronous parallel batched updates)
5. **Proximal Policy Optimization (PPO)** (GAE advantages + Clipped surrogate objective)

We train all agents on the standard **Gymnasium CartPole-v1** environment to analyze their convergence rates and training stability.

---
## System Architecture Recap
Before diving into code, let's recall the difference in information flow between the algorithms:

### 1. One-Step Actor-Critic (TD) Flow
<img src="./images/one_step_ac_info_flow.svg" width="650" alt="One-Step AC">

### 2. Synchronous Parallel A2C Flow
<img src="./images/a2c_synchronous_architecture.svg" width="650" alt="A2C Parallel Architecture">

Let's import our dependencies and configure reproducibility.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt
import random

# Set seeds for reproducibility
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)


## 1. Shared Network Architectures
We define the Actor (Policy) and Critic (Value) network architectures. In deep RL, the Actor maps state observations to action probabilities, while the Critic estimates the state-value function $V(s)$ to act as a baseline or bootstrap target.


In [ ]:
class ActorNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim=64):
        super(ActorNetwork, self).__init__()
        self.fc1 = nn.Linear(state_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, action_dim)
        
    def forward(self, state):
        x = F.relu(self.fc1(state))
        x = F.relu(self.fc2(x))
        action_logits = self.fc3(x)
        # Return categorical action probabilities
        return F.softmax(action_logits, dim=-1)

class CriticNetwork(nn.Module):
    def __init__(self, state_dim, hidden_dim=64):
        super(CriticNetwork, self).__init__()
        self.fc1 = nn.Linear(state_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, 1)
        
    def forward(self, state):
        x = F.relu(self.fc1(state))
        x = F.relu(self.fc2(x))
        return self.fc3(x)


## 2. REINFORCE (Monte Carlo Policy Gradient)
REINFORCE calculates the full episodic return $G_t = \sum_{k=0}^{\infty} \gamma^k R_{t+k+1}$ to scale the policy gradient updates. Since it uses un-bootstrapped returns, it has **zero bias** but **high variance**.


In [ ]:
class REINFORCEAgent:
    def __init__(self, state_dim, action_dim, lr=0.003, gamma=0.99):
        self.actor = ActorNetwork(state_dim, action_dim)
        self.optimizer = optim.Adam(self.actor.parameters(), lr=lr)
        self.gamma = gamma
        
    def select_action(self, state):
        state_t = torch.FloatTensor(state)
        probs = self.actor(state_t)
        dist = torch.distributions.Categorical(probs)
        action = dist.sample()
        return action.item(), dist.log_prob(action)
        
    def update(self, rewards, log_probs):
        # Calculate discounted returns backwards
        returns = []
        G = 0
        for r in reversed(rewards):
            G = r + self.gamma * G
            returns.insert(0, G)
            
        returns = torch.FloatTensor(returns)
        # Normalize returns to stabilize variance
        returns = (returns - returns.mean()) / (returns.std() + 1e-8)
        
        loss = 0
        for log_prob, G in zip(log_probs, returns):
            loss += -log_prob * G
            
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()


## 3. REINFORCE with Baseline
To reduce the high variance of REINFORCE, we introduce a Critic network as a baseline $V(s)$. The gradient is scaled by the advantage estimate $A_t = G_t - V(S_t)$.


In [ ]:
class REINFORCEWithBaselineAgent:
    def __init__(self, state_dim, action_dim, lr_actor=0.003, lr_critic=0.01, gamma=0.99):
        self.actor = ActorNetwork(state_dim, action_dim)
        self.critic = CriticNetwork(state_dim)
        self.optimizer_actor = optim.Adam(self.actor.parameters(), lr=lr_actor)
        self.optimizer_critic = optim.Adam(self.critic.parameters(), lr=lr_critic)
        self.gamma = gamma
        
    def select_action(self, state):
        state_t = torch.FloatTensor(state)
        probs = self.actor(state_t)
        dist = torch.distributions.Categorical(probs)
        action = dist.sample()
        return action.item(), dist.log_prob(action)
        
    def update(self, states, rewards, log_probs):
        returns = []
        G = 0
        for r in reversed(rewards):
            G = r + self.gamma * G
            returns.insert(0, G)
            
        states_t = torch.FloatTensor(np.array(states))
        returns_t = torch.FloatTensor(returns).unsqueeze(1)
        
        # Compute baseline values and advantages
        state_values = self.critic(states_t)
        advantages = (returns_t - state_values).detach().squeeze(1)
        
        # Actor loss
        actor_loss = 0
        for log_prob, A in zip(log_probs, advantages):
            actor_loss += -log_prob * A
            
        # Critic loss (MSE loss between baseline and Monte Carlo return)
        critic_loss = F.mse_loss(state_values, returns_t)
        
        # Update Actor
        self.optimizer_actor.zero_grad()
        actor_loss.backward()
        self.optimizer_actor.step()
        
        # Update Critic
        self.optimizer_critic.zero_grad()
        critic_loss.backward()
        self.optimizer_critic.step()


## 4. One-Step Actor-Critic (TD)
One-Step Actor-Critic bootstraps immediately after a single transition. It updates policy and value weights online at each step using the TD error $\delta_t = R_{t+1} + \gamma V(S_{t+1}) - V(S_t)$ as the advantage.


In [ ]:
class ActorCriticAgent:
    def __init__(self, state_dim, action_dim, lr_actor=0.003, lr_critic=0.01, gamma=0.99):
        self.actor = ActorNetwork(state_dim, action_dim)
        self.critic = CriticNetwork(state_dim)
        self.optimizer_actor = optim.Adam(self.actor.parameters(), lr=lr_actor)
        self.optimizer_critic = optim.Adam(self.critic.parameters(), lr=lr_critic)
        self.gamma = gamma
        
    def select_action(self, state):
        state_t = torch.FloatTensor(state)
        probs = self.actor(state_t)
        dist = torch.distributions.Categorical(probs)
        action = dist.sample()
        return action.item(), dist.log_prob(action)
        
    def update(self, state, action_log_prob, reward, next_state, done):
        state_t = torch.FloatTensor(state)
        next_state_t = torch.FloatTensor(next_state)
        
        # Estimate V(S_t) and V(S_t+1)
        v_s = self.critic(state_t)
        v_next = self.critic(next_state_t) if not done else torch.zeros(1)
        
        # TD error (Advantage)
        td_target = reward + self.gamma * v_next
        delta = td_target - v_s
        
        # Compute losses
        actor_loss = -action_log_prob * delta.item()
        critic_loss = F.mse_loss(v_s, td_target.detach())
        
        # Update Actor
        self.optimizer_actor.zero_grad()
        actor_loss.backward()
        self.optimizer_actor.step()
        
        # Update Critic
        self.optimizer_critic.zero_grad()
        critic_loss.backward()
        self.optimizer_critic.step()


## 5. Advantage Actor-Critic (A2C)
A2C gathers batches of trajectories synchronously across multiple environments, computes $T$-step rollout targets, and performs batched matrix updates on the GPU/CPU to stabilize deep network parameter convergence.


In [ ]:
class A2CAgent:
    def __init__(self, state_dim, action_dim, lr=0.003, gamma=0.99):
        # Define networks (A2C commonly uses separate Actor/Critic optimizers)
        self.actor = ActorNetwork(state_dim, action_dim)
        self.critic = CriticNetwork(state_dim)
        self.optimizer_actor = optim.Adam(self.actor.parameters(), lr=lr)
        self.optimizer_critic = optim.Adam(self.critic.parameters(), lr=lr)
        self.gamma = gamma
        
    def select_actions(self, states):
        # Batched action selection for parallel workers
        states_t = torch.FloatTensor(np.array(states))
        probs = self.actor(states_t)
        dist = torch.distributions.Categorical(probs)
        actions = dist.sample()
        return actions.numpy(), dist.log_prob(actions)
        
    def update(self, states, actions, rewards, next_states, dones, log_probs):
        states_t = torch.FloatTensor(np.array(states))
        next_states_t = torch.FloatTensor(np.array(next_states))
        rewards_t = torch.FloatTensor(rewards)
        dones_t = torch.FloatTensor(dones)
        log_probs_t = torch.stack(log_probs)
        
        # Bootstrapped values
        v_s = self.critic(states_t).squeeze(-1)
        v_next = self.critic(next_states_t).squeeze(-1)
        
        # Targets and advantages (1-step rollout batch for simplicity)
        targets = rewards_t + self.gamma * v_next * (1 - dones_t)
        advantages = targets - v_s
        
        # Losses
        actor_loss = -(log_probs_t * advantages.detach()).mean()
        critic_loss = F.mse_loss(v_s, targets.detach())
        
        # Optimize Actor
        self.optimizer_actor.zero_grad()
        actor_loss.backward()
        self.optimizer_actor.step()
        
        # Optimize Critic
        self.optimizer_critic.zero_grad()
        critic_loss.backward()
        self.optimizer_critic.step()


## 6. Proximal Policy Optimization (PPO)
PPO collects rollouts of length $T$, computes advantages using GAE (Generalized Advantage Estimation), and optimizes a Clipped Surrogate Objective over multiple epochs on shuffled mini-batches of data.


In [ ]:
class PPOAgent:
    def __init__(self, state_dim, action_dim, lr=0.0003, gamma=0.99, lmbda=0.95, eps_clip=0.2, K_epochs=4):
        # Shared backbone network setup (using Actor/Critic with shared weights is also common, but separate is easier to tune)
        self.actor = ActorNetwork(state_dim, action_dim)
        self.critic = CriticNetwork(state_dim)
        self.optimizer_actor = optim.Adam(self.actor.parameters(), lr=lr)
        self.optimizer_critic = optim.Adam(self.critic.parameters(), lr=lr)
        self.gamma = gamma
        self.lmbda = lmbda
        self.eps_clip = eps_clip
        self.K_epochs = K_epochs
        
    def select_action(self, state):
        state_t = torch.FloatTensor(state)
        probs = self.actor(state_t)
        dist = torch.distributions.Categorical(probs)
        action = dist.sample()
        return action.item(), dist.log_prob(action)
        
    def update(self, states, actions, old_log_probs, rewards, dones):
        # Convert to tensors
        states_t = torch.FloatTensor(np.array(states))
        actions_t = torch.LongTensor(actions)
        old_log_probs_t = torch.FloatTensor(old_log_probs)
        
        # 1. Compute state values
        state_values = self.critic(states_t).squeeze(-1).detach().numpy()
        
        # 2. Compute GAE advantages & returns
        advantages = []
        gae = 0
        # Append next state value 0 to values list
        values = list(state_values) + [0]
        for t in reversed(range(len(rewards))):
            delta = rewards[t] + self.gamma * values[t+1] * (1 - dones[t]) - values[t]
            gae = delta + self.gamma * self.lmbda * (1 - dones[t]) * gae
            advantages.insert(0, gae)
            
        advantages_t = torch.FloatTensor(advantages)
        # Calculate target returns
        returns_t = advantages_t + torch.FloatTensor(state_values)
        # Normalize advantages
        advantages_t = (advantages_t - advantages_t.mean()) / (advantages_t.std() + 1e-8)
        
        # 3. K epochs of mini-batch gradient updates
        for _ in range(self.K_epochs):
            # Calculate current policy log probabilities and values
            probs = self.actor(states_t)
            dist = torch.distributions.Categorical(probs)
            log_probs = dist.log_prob(actions_t)
            entropy = dist.entropy()
            
            v_s = self.critic(states_t).squeeze(-1)
            
            # Ratio r_t(theta)
            ratios = torch.exp(log_probs - old_log_probs_t)
            
            # Clipped surrogate loss
            surr1 = ratios * advantages_t
            surr2 = torch.clamp(ratios, 1.0 - self.eps_clip, 1.0 + self.eps_clip) * advantages_t
            actor_loss = -torch.min(surr1, surr2).mean() - 0.01 * entropy.mean()
            
            # Value loss
            critic_loss = F.mse_loss(v_s, returns_t)
            
            # Update Actor
            self.optimizer_actor.zero_grad()
            actor_loss.backward()
            self.optimizer_actor.step()
            
            # Update Critic
            self.optimizer_critic.zero_grad()
            critic_loss.backward()
            self.optimizer_critic.step()


## 7. Comparative Performance Evaluation
We train each agent under matching configurations on `CartPole-v1` and record their reward progress to visualize convergence rate and variance curves.


In [ ]:
env = gym.make('CartPole-v1')
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n
num_episodes = 250
max_steps = 500

all_rewards = {}

# -------------------- Run REINFORCE --------------------
print('Training REINFORCE Agent... ')
reinforce_agent = REINFORCEAgent(state_dim, action_dim)
reinforce_rewards = []
for ep in range(num_episodes):
    state = env.reset(seed=seed+ep)[0] if hasattr(env, 'reset') else env.reset()
    ep_rewards = []
    log_probs = []
    for step in range(max_steps):
        action, log_prob = reinforce_agent.select_action(state)
        state, reward, done, truncated, info = env.step(action)
        ep_rewards.append(reward)
        log_probs.append(log_prob)
        if done or (truncated if 'truncated' in locals() else False):
            break
    reinforce_agent.update(ep_rewards, log_probs)
    reinforce_rewards.append(sum(ep_rewards))
all_rewards['REINFORCE'] = reinforce_rewards

# -------------------- Run REINFORCE with Baseline --------------------
print('Training REINFORCE with Baseline Agent... ')
baseline_agent = REINFORCEWithBaselineAgent(state_dim, action_dim)
baseline_rewards = []
for ep in range(num_episodes):
    state = env.reset(seed=seed+ep)[0] if hasattr(env, 'reset') else env.reset()
    ep_rewards = []
    states = []
    log_probs = []
    for step in range(max_steps):
        action, log_prob = baseline_agent.select_action(state)
        states.append(state)
        state, reward, done, truncated, info = env.step(action)
        ep_rewards.append(reward)
        log_probs.append(log_prob)
        if done or (truncated if 'truncated' in locals() else False):
            break
    baseline_agent.update(states, ep_rewards, log_probs)
    baseline_rewards.append(sum(ep_rewards))
all_rewards['REINFORCE w/ Baseline'] = baseline_rewards

# -------------------- Run One-Step Actor-Critic --------------------
print('Training One-Step Actor-Critic Agent... ')
ac_agent = ActorCriticAgent(state_dim, action_dim)
ac_rewards = []
for ep in range(num_episodes):
    state = env.reset(seed=seed+ep)[0] if hasattr(env, 'reset') else env.reset()
    ep_reward = 0
    for step in range(max_steps):
        action, log_prob = ac_agent.select_action(state)
        next_state, reward, done, truncated, info = env.step(action)
        is_done = done or (truncated if 'truncated' in locals() else False)
        ac_agent.update(state, log_prob, reward, next_state, is_done)
        state = next_state
        ep_reward += reward
        if is_done:
            break
    ac_rewards.append(ep_reward)
all_rewards['Actor-Critic (1-Step TD)'] = ac_rewards

# -------------------- Run A2C --------------------
print('Training A2C Agent... ')
a2c_agent = A2CAgent(state_dim, action_dim)
a2c_rewards = []
for ep in range(num_episodes):
    state = env.reset(seed=seed+ep)[0] if hasattr(env, 'reset') else env.reset()
    ep_rewards = []
    states = []
    actions = []
    next_states = []
    dones = []
    log_probs = []
    for step in range(max_steps):
        action, log_prob = a2c_agent.select_actions([state])
        states.append(state)
        actions.append(action[0])
        log_probs.append(log_prob[0])
        
        next_state, reward, done, truncated, info = env.step(action[0])
        is_done = done or (truncated if 'truncated' in locals() else False)
        
        rewards = [reward]
        next_states.append(next_state)
        dones.append(is_done)
        
        a2c_agent.update([state], [action[0]], [reward], [next_state], [float(is_done)], [log_prob[0]])
        state = next_state
        ep_rewards.append(reward)
        if is_done:
            break
    a2c_rewards.append(sum(ep_rewards))
all_rewards['A2C (Batched)'] = a2c_rewards

# -------------------- Run PPO --------------------
print('Training PPO Agent... ')
ppo_agent = PPOAgent(state_dim, action_dim)
ppo_rewards = []
for ep in range(num_episodes):
    state = env.reset(seed=seed+ep)[0] if hasattr(env, 'reset') else env.reset()
    ep_rewards = []
    states = []
    actions = []
    dones = []
    log_probs = []
    for step in range(max_steps):
        action, log_prob = ppo_agent.select_action(state)
        states.append(state)
        actions.append(action)
        log_probs.append(log_prob.item())
        
        next_state, reward, done, truncated, info = env.step(action)
        is_done = done or (truncated if 'truncated' in locals() else False)
        
        ep_rewards.append(reward)
        dones.append(float(is_done))
        state = next_state
        if is_done:
            break
    ppo_agent.update(states, actions, log_probs, ep_rewards, dones)
    ppo_rewards.append(sum(ep_rewards))
all_rewards['PPO (Clipped)'] = ppo_rewards
print('Training Completed.')


## 8. Learning Curve Visualization
We plot the rolling average rewards of each agent to evaluate their relative data efficiency, final performance caps, and variance/stability levels.


In [ ]:
plt.figure(figsize=(10, 6))
for name, rewards in all_rewards.items():
    # Compute running mean of 10 episodes to smooth plots
    smoothed = np.convolve(rewards, np.ones(10)/10, mode='valid')
    plt.plot(smoothed, label=name)
plt.xlabel('Episodes')
plt.ylabel('Smoothed Episode Reward (Window=10)')
plt.title('Policy Gradient & PPO Performance Comparison on CartPole-v1')
plt.legend()
plt.grid(True)
plt.show()
